In [0]:
# drop all tables from brazilian_ecommerce database
spark.sql(f"DROP TABLE IF EXISTS brazilian_ecommerce.prod.bronze_orders")

In [0]:
BASE_PATH = "/Volumes/brazilian_ecommerce/prod/raw_data"
CATALOG = "brazilian_ecommerce"
SCHEMA = "prod"

# csv file name -> (bronze table name, comment)
TABLES = {
    "olist_order_reviews_dataset": ("bronze_orders_review", "orders review raw table"),
    "olist_orders_dataset": ("bronze_orders", "orders raw table"),
    "olist_order_items_dataset": ("bronze_order_items", "order items raw table"),
    "olist_products_dataset": ("bronze_products", "products raw table"),
    "olist_customers_dataset": ("bronze_customers", "customers raw table"),
    "olist_sellers_dataset": ("bronze_sellers", "sellers raw table"),
    "olist_order_payments_dataset": ("bronze_order_payments", "order payments raw table"),
    "olist_geolocation_dataset": ("bronze_geolocation", "geolocation raw table"),
    "product_category_name_translation": ("bronze_category_translation", "category translation raw table"),
}

def load_to_bronze(csv_name: str, bronze_table: str, comment: str) -> None:
    csv_path = f"{BASE_PATH}/{csv_name}.csv"
    df = (
        spark.read
        .option("header", True)
        .csv(csv_path)
    )
    full_table_name = f"{CATALOG}.{SCHEMA}.{bronze_table}"
    (
        df.write
        .mode("overwrite")
        .option("comment", comment)
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )
    print(f"Loaded {csv_path} -> {full_table_name} ({df.count()} rows)")

for csv_name, (bronze_table, comment) in TABLES.items():
    load_to_bronze(csv_name, bronze_table, comment)